# 1. Setup

In [1]:
import os
import shutil

def organize_images(input_dir, output_dir):
    # List all files in the input directory
    files = os.listdir(input_dir)
    
    for file in files:
        # Check if the file is an image (you can add more extensions if needed)
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')):
            # Extract the 'word' part from the filename
            word = ''.join([i for i in file if not i.isdigit()]).split('.')[0]
            
            # Create a new directory for the word in the output directory if it doesn't exist
            word_dir = os.path.join(output_dir, word)
            if not os.path.exists(word_dir):
                os.makedirs(word_dir)
            
            # Copy the file to the corresponding directory in the output folder
            shutil.copy(os.path.join(input_dir, file), os.path.join(word_dir, file))


# Specify the input directory containing the images
input_directory = "/kaggle/input/indian-sign-language-image-dataset/NoFilterModeDataSet"

# Specify the output directory where organized images will be placed
output_directory = "output"

# Create the output directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

# Organize the images
organize_images(input_directory, output_directory)

In [2]:
!pip install easyfsl -q
!pip install split-folders[full] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.0 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd

import cv2
from PIL import Image


import torch
from torch import nn, optim
from torch.utils.data import DataLoader

from torchvision.models import resnet18
from torchvision import datasets, transforms

from tqdm import tqdm

import random

from easyfsl.datasets import WrapFewShotDataset
from easyfsl.samplers import TaskSampler
from easyfsl.utils import plot_images, sliding_average

import splitfolders

# 2. Config

In [4]:
random_seed = 0
np.random.seed(random_seed)
torch.manual_seed(random_seed)
random.seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [5]:
N_WAY = 5  # Number of classes in a task
N_SHOT = 5  # Number of images per class in the support set
N_QUERY = 10  # Number of images per class in the query set
N_EVALUATION_TASKS = 500

DEVICE = "cuda"
n_workers = 4

image_size = 64  # Adjust as needed

# 3. Data Prepration

In [6]:
input_folder='output'
output_folder='dataset'

splitfolders.ratio(input_folder, output=output_folder,
    seed=1337, ratio=(.2, .2, .6), group_prefix=None, move=False) # default values

Copying files: 5000 files [00:00, 6617.01 files/s]


# 3.1. Data Preprocessing

In [7]:
def edge_enhancement(img):
    """
    Custom function to perform edge enhancement using OpenCV.
    Args:
        img: PIL Image
    Returns:
        PIL Image with enhanced edges
    """
    # Convert PIL Image to NumPy array
    img_np = np.array(img)

    # Convert to grayscale if needed
    if len(img_np.shape) == 3:  # If the image has 3 channels (RGB)
        img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    # Apply Gaussian blur
    blurred = cv2.GaussianBlur(img_np, (5, 5), 2)

    # Apply adaptive thresholding
    thresholded = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )

    # Apply Otsu's thresholding
    _, processed = cv2.threshold(
        thresholded, 70, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Convert back to PIL Image
    processed_pil = Image.fromarray(processed)

    # Convert back to 3 channels if needed
    if len(img_np.shape) == 3:
        processed_pil = processed_pil.convert("RGB")

    return processed_pil



In [8]:
image_transform = transforms.Compose(
    [
        transforms.Lambda(edge_enhancement),  # Apply custom edge enhancement
        transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels if needed
        transforms.RandomResizedCrop(image_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]
)

# 3.2 Load DataSet

In [9]:
# Load the dataset
train_set = datasets.ImageFolder(
    root="./dataset/train",
    transform=image_transform
)

test_set = datasets.ImageFolder(
    root="./dataset/test",
    transform=image_transform
)

val_set = datasets.ImageFolder(
    root="./dataset/val",
    transform=image_transform
)

# Example usage
print(f"Number of training samples: {len(train_set)}")
print(f"Number of test samples: {len(test_set)}")
print(f"Number of val samples: {len(val_set)}")

Number of training samples: 1000
Number of test samples: 3000
Number of val samples: 1000


In [10]:
train_set=WrapFewShotDataset(dataset=train_set)
test_set=WrapFewShotDataset(dataset=test_set)
val_set=WrapFewShotDataset(dataset=val_set)

Scrolling dataset's labels...: 100%|██████████| 1000/1000 [00:03<00:00, 299.30it/s]


In [11]:
train_sampler = TaskSampler(
    train_set, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_EVALUATION_TASKS
)

test_sampler = TaskSampler(
    test_set, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_EVALUATION_TASKS
)

val_sampler = TaskSampler(
    val_set, n_way=N_WAY, n_shot=N_SHOT, n_query=N_QUERY, n_tasks=N_EVALUATION_TASKS
)

In [12]:
train_loader = DataLoader(
    train_set,
    batch_sampler=train_sampler,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=train_sampler.episodic_collate_fn,
)
test_loader = DataLoader(
    test_set,
    batch_sampler=test_sampler,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=test_sampler.episodic_collate_fn,
)
val_loader = DataLoader(
    val_set,
    batch_sampler=val_sampler,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=val_sampler.episodic_collate_fn,
)

# 4. Model Creation

In [13]:
from easyfsl.methods import PrototypicalNetworks, FewShotClassifier
from easyfsl.modules import resnet12

In [14]:
# class PrototypicalNetworks(nn.Module):
#     def __init__(self, backbone: nn.Module):
#         super(PrototypicalNetworks, self).__init__()
#         self.backbone = backbone

#     def forward(
#         self,
#         support_images: torch.Tensor,
#         support_labels: torch.Tensor,
#         query_images: torch.Tensor,
#     ) -> torch.Tensor:
#         """
#         Predict query labels using labeled support images.
#         """
#         # Extract the features of support and query images
#         z_support = self.backbone.forward(support_images)
#         z_query = self.backbone.forward(query_images)

#         # Infer the number of different classes from the labels of the support set
#         n_way = len(torch.unique(support_labels))
#         # Prototype i is the mean of all instances of features corresponding to labels == i
#         z_proto = torch.cat(
#             [
#                 z_support[torch.nonzero(support_labels == label)].mean(0)
#                 for label in range(n_way)
#             ]
#         )

#         # Compute the euclidean distance from queries to prototypes
#         dists = torch.cdist(z_query, z_proto)

#         # And here is the super complicated operation to transform those distances into classification scores!
#         scores = -dists
#         return scores


# convolutional_network = resnet18(pretrained=True)
# convolutional_network.fc = nn.Flatten()
# print(convolutional_network)

# few_shot_classifier = PrototypicalNetworks(convolutional_network).cuda()

In [15]:


#from easyfsl.methods import PrototypicalNetworks, FewShotClassifier
#from easyfsl.modules import resnet12


#convolutional_network = resnet12()
#few_shot_classifier = PrototypicalNetworks(convolutional_network).to(DEVICE)



In [16]:
import torch
import torch.nn as nn
from torchvision.models import resnet18
from easyfsl.methods import PrototypicalNetworks

def get_resnet_backbone(pretrained=True):
    # Load ResNet18 backbone
    backbone = resnet18(pretrained=True)
    
    # Remove the final fully connected layer and add Global Average Pooling
    backbone = nn.Sequential(
        *list(backbone.children())[:-2],  # Remove the last two layers (avgpool and fc)
        nn.AdaptiveAvgPool2d((1, 1)),     # Add Global Average Pooling
        nn.Flatten(),                     # Flatten to 1D vector
    )
    
    # Freeze early layers (optional, adjust as needed)
    for name, param in backbone.named_parameters():
        if 'layer1' in name or 'conv1' in name:  # Freeze first few layers
            param.requires_grad = False
    return backbone

# Initialize the backbone and FSL model
backbone = get_resnet_backbone(pretrained=True)
few_shot_classifier = PrototypicalNetworks(backbone).to(DEVICE)

print("Model created successfully!")

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


Model created successfully!


# Episodic training

In [17]:
from torch.optim import SGD, Optimizer
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.tensorboard import SummaryWriter
from pathlib import Path

LOSS_FUNCTION = nn.CrossEntropyLoss()

n_epochs = 200
scheduler_milestones = [120, 160]
scheduler_gamma = 0.1
learning_rate = 1e-2
tb_logs_dir = Path(".")

train_optimizer = SGD(
    few_shot_classifier.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4
)
train_scheduler = MultiStepLR(
    train_optimizer,
    milestones=scheduler_milestones,
    gamma=scheduler_gamma,
)

tb_writer = SummaryWriter(log_dir=str(tb_logs_dir))

In [18]:
def training_epoch(
    model: FewShotClassifier, data_loader: DataLoader, optimizer: Optimizer
):
    all_loss = []
    model.train()
    with tqdm(
        enumerate(data_loader), total=len(data_loader), desc="Training"
    ) as tqdm_train:
        for episode_index, (
            support_images,
            support_labels,
            query_images,
            query_labels,
            _,
        ) in tqdm_train:
            optimizer.zero_grad()
            model.process_support_set(
                support_images.to(DEVICE), support_labels.to(DEVICE)
            )
            classification_scores = model(query_images.to(DEVICE))

            loss = LOSS_FUNCTION(classification_scores, query_labels.to(DEVICE))
            loss.backward()
            optimizer.step()

            all_loss.append(loss.item())

            tqdm_train.set_postfix(loss=mean(all_loss))

    return mean(all_loss)

In [19]:
import copy
from pathlib import Path
import random
from statistics import mean

import numpy as np
import torch
from torch import nn
from tqdm import tqdm

In [20]:
from easyfsl.utils import evaluate


best_state = few_shot_classifier.state_dict()
best_validation_accuracy = 0.0
for epoch in range(n_epochs):
    print(f"Epoch {epoch}")
    average_loss = training_epoch(few_shot_classifier, train_loader, train_optimizer)
    validation_accuracy = evaluate(
        few_shot_classifier, val_loader, device=DEVICE, tqdm_prefix="Validation"
    )

    if validation_accuracy > best_validation_accuracy:
        best_validation_accuracy = validation_accuracy
        best_state = copy.deepcopy(few_shot_classifier.state_dict())
        # state_dict() returns a reference to the still evolving model's state so we deepcopy
        # https://pytorch.org/tutorials/beginner/saving_loading_models
        print("Ding ding ding! We found a new best model!")

    tb_writer.add_scalar("Train/loss", average_loss, epoch)
    tb_writer.add_scalar("Val/acc", validation_accuracy, epoch)

    # Warn the scheduler that we did an epoch
    # so it knows when to decrease the learning rate
    train_scheduler.step()

Epoch 0


Validation: 100%|██████████| 500/500 [01:13<00:00,  6.77it/s, accuracy=0.822]

Ding ding ding! We found a new best model!
Epoch 1



Validation: 100%|██████████| 500/500 [01:15<00:00,  6.61it/s, accuracy=0.869]

Ding ding ding! We found a new best model!
Epoch 2


...


Validation: 100%|██████████| 500/500 [01:11<00:00,  6.98it/s, accuracy=0.972]

Epoch 198



Validation: 100%|██████████| 500/500 [01:11<00:00,  6.98it/s, accuracy=0.97]

Epoch 199



Validation: 100%|██████████| 500/500 [01:11<00:00,  6.97it/s, accuracy=0.97]


In [21]:
few_shot_classifier.load_state_dict(best_state)

<All keys matched successfully>

# Eval

In [22]:
accuracy = evaluate(few_shot_classifier, test_loader, device=DEVICE)
print(f"Average accuracy : {(100 * accuracy):.2f} %")

100%|██████████| 500/500 [01:11<00:00,  6.97it/s, accuracy=0.964]

Average accuracy : 96.36 %


In [23]:
import torch
from pathlib import Path

# Save the model's state dictionary and metadata
checkpoint = {
    "model_state_dict": few_shot_classifier.state_dict(),  # Model weights
    "class_labels": ["Aboard","All_Gone","Baby","Beside","Book","Bowl","Bridge","Camp","Cartridge","Eight","Five","Fond","Four","Friend","Glove","Hang","High","House","How_Many","I_or_Me","Man","Marry","Meat","Medal","Mid_Day","Middle","Money","Money","Moon","Mother","Nine","One","Opposite","Prison","Ring","Rose","See","Seven","Short","Six","Superior","Ten","Thick","Thin","Three","Tobacco","Two","Up","Watch","Write","You"],  # Example class labels
    "input_size": (3, 64, 64),  # Example input size (for preprocessing)
}


# Save the checkpoint to a file
model_path = Path("classification_model.pth")
torch.save(checkpoint, model_path)
print(f"Model saved to {model_path}")

Model saved to classification_model.pth


In [24]:
import torch
from easyfsl.methods import PrototypicalNetworks
# from easyfsl.modules import resnet12

# Define the model architecture
# convolutional_network = resnet12()
# model = PrototypicalNetworks(convolutional_network)
import torch
import torch.nn as nn
from torchvision.models import resnet18
from easyfsl.methods import PrototypicalNetworks

def get_resnet_backbone(pretrained=True):
    # Load ResNet18 backbone
    backbone = resnet18(pretrained=True)
    
    # Remove the final fully connected layer and add Global Average Pooling
    backbone = nn.Sequential(
        *list(backbone.children())[:-2],  # Remove the last two layers (avgpool and fc)
        nn.AdaptiveAvgPool2d((1, 1)),     # Add Global Average Pooling
        nn.Flatten(),                     # Flatten to 1D vector
    )
    
    # Freeze early layers (optional, adjust as needed)
    for name, param in backbone.named_parameters():
        if 'layer1' in name or 'conv1' in name:  # Freeze first few layers
            param.requires_grad = False
    return backbone

# Initialize the backbone and FSL model
backbone = get_resnet_backbone(pretrained=True)
model = PrototypicalNetworks(backbone).to(DEVICE)

# Load the checkpoint
checkpoint = torch.load("classification_model.pth", map_location=torch.device("cuda"),weights_only=False)  # Use "cuda" for GPU
model.load_state_dict(checkpoint["model_state_dict"])

# Move the model to the appropriate device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Set the model to evaluation mode
model.eval()

# Example class labels and input size
class_labels = checkpoint["class_labels"]
input_size = checkpoint["input_size"]

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [25]:
accuracy = evaluate(model, test_loader, device=DEVICE)
print(f"Average accuracy : {(100 * accuracy):.2f} %")

100%|██████████| 500/500 [01:12<00:00,  6.90it/s, accuracy=0.966]

Average accuracy : 96.60 %
